In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split , cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor , StackingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score , mean_absolute_error
from sklearn.compose import ColumnTransformer , TransformedTargetRegressor
from sklearn.pipeline import Pipeline 
from sklearn.preprocessing import StandardScaler , OneHotEncoder , OrdinalEncoder , PowerTransformer
from sklearn.base import BaseEstimator , TransformerMixin
import mlflow
import dagshub

In [2]:
dagshub.init(repo_owner="mridul0010" , repo_name="food-delivery-time-prediction" , mlflow=True)

Accessing as mridul0010

Initialized MLflow to track repo "mridul0010/food-delivery-time-prediction"

Repository mridul0010/food-delivery-time-prediction initialized!

In [3]:
mlflow.set_tracking_uri("https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow")

In [4]:
mlflow.set_experiment("Final Estimation")

2026/07/05 22:52:01 INFO mlflow.tracking.fluent: Experiment with name 'Final Estimation' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/e62bf0a574b149f2b0faefc0f9ffa583', creation_time=1783272123676, effective_trace_archival_retention=None, experiment_id='6', last_update_time=1783272123676, lifecycle_stage='active', name='Final Estimation', tags={}, trace_location=None, workspace='default'>

In [5]:
df = pd.read_csv('../data/Cleaned Delivery Dataset.csv')

In [6]:
pd.set_option('display.max_columns' , None)

In [7]:
df.drop(columns=['Delivery_Agent'] , inplace=True)

In [8]:
df['Order_Datetime'] = pd.to_datetime(df['Order_Datetime'])
df['Pickup_Datetime'] = pd.to_datetime(df['Pickup_Datetime'])

In [9]:
class FeatureEngineering(BaseEstimator, TransformerMixin):
    
    def __init__(self):
        self.rating_bins = None

    def fit(self, X, y=None):
        X = X.copy()
        
        # Store bins for consistent transformation
        self.rating_bins = pd.qcut(
            X['Delivery_person_Ratings'],
            q=3,
            retbins=True,
            duplicates='drop'
        )[1]
        
        return self

    def transform(self, X):
        X = X.copy()

        # Datetime conversion
        X['Order_Datetime'] = pd.to_datetime(X['Order_Datetime'], errors='coerce')
        X['Pickup_Datetime'] = pd.to_datetime(X['Pickup_Datetime'], errors='coerce')

        # Rating group
        X["delivery_rating_group"] = pd.cut(
            X['Delivery_person_Ratings'],
            bins=self.rating_bins,
            labels=['Low', 'Medium', 'High'],
            include_lowest=True
        )

        # Age group
        X["age_group"] = pd.cut(
            X['Delivery_person_Age'],
            bins=[14, 25, 35, 60],
            labels=['Young', 'Adult', 'Senior']
        )

        # Distance group
        X["distance_group"] = pd.cut(
            X['distance_km'],
            bins=[0, 5, 10, 25],
            labels=['Short Distance', 'Medium Distance', 'Long Distance']
        )

        # Time features
        X['Prep_Time(min)'] = (
            X['Pickup_Datetime'] - X['Order_Datetime']
        ).dt.total_seconds() / 60

        X['Order_hour'] = X['Order_Datetime'].dt.hour
        X['Order_day'] = X['Order_Datetime'].dt.day_name()

        X['isWeekend'] = X['Order_day'].isin(["Saturday", "Sunday"]).astype(int)

        X['Time_Of_Day'] = pd.cut(
            X['Order_hour'],
            bins=[0, 6, 12, 18, 24],
            labels=["Night", "Morning", "Afternoon", "Evening"],
            include_lowest=True
        )

        # Drop raw datetime
        X = X.drop(columns=['Order_Datetime', 'Pickup_Datetime'], errors='ignore')

        return X

    def get_feature_names_out(self , input_features = None):
        return input_features

In [10]:
X = df.drop(columns=['Time_taken (min)'])
y = df['Time_taken (min)']

In [11]:
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2 , random_state=42)

In [12]:
X

,City,Zone,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,distance_km,Order_Datetime,Pickup_Datetime,Weather_conditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City_Type
0,Dehradun,Zone17,36,4.2,30.327968,78.046106,30.397968,78.116106,10.28,2022-12-02 21:55:00,2022-12-02 22:10:00,Fog,Jam,Good,Snack,motorcycle,3,No,Metropolitian
1,Kochi,Zone16,21,4.7,10.003064,76.307589,10.043064,76.347589,6.24,2022-02-13 14:55:00,2022-02-13 15:05:00,Stormy,High,Average,Meal,motorcycle,1,No,Metropolitian
2,Pune,Zone13,23,4.7,18.562450,73.916619,18.652450,74.006619,13.79,2022-04-03 17:30:00,2022-04-03 17:40:00,Sandstorms,Medium,Average,Drinks,scooter,1,No,Metropolitian
3,Ludhiana,Zone15,34,4.3,30.899584,75.809346,30.919584,75.829346,2.93,2022-02-13 09:20:00,2022-02-13 09:30:00,Sandstorms,Low,poor,Buffet,motorcycle,0,No,Metropolitian
4,Kanpur,Zone14,24,4.7,26.463504,80.372929,26.593504,80.502929,19.40,2022-02-14 19:50:00,2022-02-14 20:05:00,Fog,Jam,Average,Snack,scooter,1,No,Metropolitian
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39637,Bangalore,Zone16,28,4.9,13.029198,77.570997,13.059198,77.600997,4.66,2022-03-30 21:55:00,2022-03-30 22:05:00,Sandstorms,Jam,Average,Meal,scooter,1,No,Metropolitian
39638,Ranchi,Zone16,35,4.2,23.371292,85.327872,23.481292,85.437872,16.60,2022-08-03 21:45:00,2022-08-03 21:55:00,Windy,Jam,Good,Drinks,motorcycle,1,No,Metropolitian
39639,Chennai,Zone08,30,4.9,13.022394,80.242439,13.052394,80.272439,4.66,2022-11-03 23:50:00,2022-11-04 00:05:00,Cloudy,Low,Average,Drinks,scooter,0,No,Metropolitian
39640,Coimbatore,Zone11,20,4.7,11.001753,76.986241,11.041753,77.026241,6.23,2022-07-03 13:35:00,2022-07-03 13:40:00,Cloudy,High,poor,Snack,motorcycle,1,No,Metropolitian


In [13]:
numerical = ['Delivery_person_Age', 'Delivery_person_Ratings', 'Restaurant_latitude',
       'Restaurant_longitude', 'Delivery_location_latitude',
       'Delivery_location_longitude', 
       'multiple_deliveries', 'distance_km', 'Prep_Time(min)', 'Order_hour',
       'isWeekend']

ohe_col = ['City', 'Zone', 'Weather_conditions',
       'Type_of_order', 'Type_of_vehicle',
       'City_Type','Order_day', 'Time_Of_Day']

ordinal_col = ['Road_traffic_density' , 'Vehicle_condition' , 'Festival',
       'delivery_rating_group' , 'age_group' , 'distance_group']

In [14]:
Road_traffic_density = ['Low', 'Medium', 'High', 'Jam']
Vehicle_condition = ['poor', 'Average', 'Good' , 'Excellent']
Festival = ['No', 'Yes']
delivery_rating_group = ['Low', 'Medium', 'High']
age_group = ['Young', 'Adult', 'Senior']         
distance_group = ['Short Distance', 'Medium Distance', 'Long Distance']

In [15]:
transformer = ColumnTransformer(
    transformers=[
        ("ohe" , OneHotEncoder(sparse_output=False , handle_unknown='ignore') , ohe_col),
        ('oe' , OrdinalEncoder(categories=[
            Road_traffic_density , Vehicle_condition,
            Festival , delivery_rating_group,
            age_group , distance_group
        ]) , ordinal_col),
        ('Scaling' , StandardScaler() , numerical)
    ],remainder='passthrough'
)

In [16]:
pipeline = Pipeline(
    [
        ("FeatureEngineering" , FeatureEngineering()),
        ("ColumnTransformer" , transformer)
    ]
) 

In [17]:
X_train_trans = pipeline.fit_transform(X_train)
X_test_trans = pipeline.transform(X_test)

In [18]:
best_xgb_params = {
    'n_estimators': 245,
    'max_depth': 23,
    'learning_rate': 0.19864909172366368,
    'subsample': 0.9180765181845632,
    'min_child_weight': 7,
    'gamma': 1.0407049003047555,
    'reg_lambda': 22.40266224283313
}

best_rf_params = {
    'n_estimators': 299,
    'max_depth': 13,
    'min_samples_split': 20,
    'min_samples_leaf': 2,
    'max_features': 0.44603640869435546,
    'bootstrap': False
}

best_rf = RandomForestRegressor(**best_rf_params)
best_xgb = XGBRegressor(**best_xgb_params)

LR = LinearRegression()

pt = PowerTransformer()

In [19]:
# stacking regressor
stacking_reg = StackingRegressor(
    estimators=[("rf",best_rf) , ("xgb",best_xgb)],
    final_estimator=LR,
    cv=5,n_jobs=-1
)

model = TransformedTargetRegressor(regressor=stacking_reg,
    transformer=pt 
)

model.fit(X_train_trans,y_train)

,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.","StackingRegre...(), n_jobs=-1)"
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",PowerTransformer()
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",None
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",None
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
Name,Type,Value
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying regressor exposes such an attribute when fit... versionadded:: 0.24,int,87
regressor_ regressor_: objectFitted regressor.,StackingRegressor,"StackingRegre...(), n_jobs=-1)"
transformer_ transformer_: objectTransformer used in :meth:`fit` and :meth:`predict`.,PowerTransformer,PowerTransformer()
,"estimators estimators: list of (str, estimator)Base estimators which will be stacked together. Each element of thelist is defined as a tuple of string (i.e. name) and an estimatorinstance. An estimator can be set to 'drop' using `set_params`.","[('rf', ...), ('xgb', ...)]"
,"final_estimator final_estimator: estimator, default=NoneA regressor which will be used to combine the base estimators.The default regressor is a :class:`~sklearn.linear_model.RidgeCV`.",LinearRegression()


In [20]:
# get the train and test predictions
y_train_pred = model.predict(X_train_trans)
y_test_pred = model.predict(X_test_trans)

# calculate the train and test mae
train_mae = mean_absolute_error(y_train,y_train_pred)
test_mae = mean_absolute_error(y_test,y_test_pred)

# calculate the r2 scores
train_r2 = r2_score(y_train,y_train_pred)
test_r2 = r2_score(y_test,y_test_pred)

In [21]:
# Combine everything into a single, comprehensive pipeline
full_cv_pipeline = Pipeline([
    ("preprocessing", pipeline),  # FeatureEngineering + ColumnTransformer
    ("model", TransformedTargetRegressor(
        regressor=stacking_reg,
        transformer=pt
    ))
])

# Run cross-validation on the RAW training data
cv_scores = cross_val_score(
    full_cv_pipeline,
    X_train,                     
    y_train,
    cv=3,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

In [22]:
print(f"Train MAE: {train_mae}")
print(f"Test MAE: {test_mae}")
print(f"Train R2: {train_r2}")
print(f"Test R2: {test_r2}")
print(f"Cross Val Scores: {-(cv_scores)}")
print(f"Mean Cross Val Scores: {- (cv_scores.mean())}")

Train MAE: 2.784845124189745
Test MAE: 2.9812223711277026
Train R2: 0.8597016901717311
Test R2: 0.8370568174261839
Cross Val Scores: [3.02902227 3.01050267 3.018059  ]
Mean Cross Val Scores: 3.0191946474018305


In [26]:
# log with mlflow
with mlflow.start_run():
    # set tags
    mlflow.set_tag("model","stacking regressor")

    # log parameters
    mlflow.log_params(stacking_reg.get_params())

    # log metrics
    mlflow.log_metric("train_mae",train_mae)
    mlflow.log_metric("test_mae",test_mae)
    mlflow.log_metric("train_r2",train_r2)
    mlflow.log_metric("test_r2",test_r2)
    mlflow.log_metric("cv_score",-(cv_scores.mean()))

    # log the stacking regressor
    mlflow.sklearn.log_model(
        sk_model=stacking_reg, 
        name="model_Stacking", 
        serialization_format="cloudpickle"
    )

2026/07/05 22:59:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run capricious-roo-456 at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/6/runs/9b133c41542a40b3b9cbd8836021213b
🧪 View experiment at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/6
